# techqa Experiment Runner — OpenRouter variant

RAG ablation sweep for the **techqa** subset of `galileo-ai/ragbench`
(IBM technical-support QA), with all LLM calls through **OpenRouter**.
Runs in parallel with the Groq techqa run and the other OpenRouter notebooks —
own `config/`, `reports/`, `temp/`, and `cache_openrouter_techqa/` dirs.

Driven by `experiment_configs/techqa_openrouter_experiment.yaml`.
**Requires** `OPENROUTER_API_KEY` (comma-separate multiple keys to rotate).
Heaviest sweep of the three subsets — watch OpenRouter credit/limit usage.


## 1. Setup & Dependencies

In [10]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq openai python-dotenv nltk pandas -q')



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports

In [11]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Point this at wherever THIS repo (rag_cust_support) lives. ---
# On Colab this is typically under your mounted Drive. Adjust if different.
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
if not PROJECT_ROOT.exists():
    # Fallback: running locally from the notebooks/ folder.
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
import experiment.experiment_runner as _er, core.registry as _reg

load_dotenv(override=True)
print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
print('core.registry   loaded from:', _reg.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'
print('HuggingFace token loaded:', bool(os.getenv('HF_TOKEN')))
print('Groq API key loaded:', bool(os.getenv('GROQ_API_KEY')))
print('OpenRouter API key loaded:', bool(os.getenv('OPENROUTER_API_KEY')))


Current directory: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support
experiment_runner loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/experiment/experiment_runner.py
core.registry   loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/core/registry.py
HuggingFace token loaded: True
Groq API key loaded: True
OpenRouter API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [24]:
EXPERIMENT_CONFIG_PATH = project_root / "experiment_configs/techqa_openrouter_validation.yaml"

experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Temp Dir:    {experiment_config.temp_dir}")
print(f"  Cache:       {experiment_config.cache}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  rag-experiments/techqa-openrouter-experiment/validation_config
  Report Dir:  rag-experiments/techqa-openrouter-experiment/reports
  Temp Dir:    rag-experiments/techqa-openrouter-experiment/temp
  Cache:       {'enabled': True, 'cache_dir': './cache_openrouter_techqa'}
  Num Queries: 100
  Parallel:    False
  Max Workers: 1

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'techqa', 'split': 'test', 'limit': 314}

Data Parser:
  Type: noop


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [25]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


In [26]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/techqa (test)...
Loaded 314 samples

✅ Data loaded successfully!
  Documents: 769 parsed documents
  Raw Data:  314 samples

First Sample:
  Question: Using cobol copybooks Sometimes, there will be errors/fields missing in typetree, while importing co...
  Documents: 5


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [27]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f'Loaded {len(configs)} RAG pipeline configurations:')
for cfg in configs:
    searches = ' + '.join(s.type.value for s in cfg.retrieval.search.searches)
    fusion = cfg.retrieval.fusion.type.value if cfg.retrieval.fusion else '-'
    rerank = cfg.retrieval.rerank.type.value if cfg.retrieval.rerank else '-'
    qx = cfg.retrieval.query_transform.type.value if cfg.retrieval.query_transform else '-'
    gc = cfg.generation.config
    model = gc.get('model') if isinstance(gc, dict) else getattr(gc, 'model', None)
    print(f'  - {cfg.name}')
    print(f'      chunking={cfg.chunking.type.value}  embed={cfg.embedding.type.value}')
    print(f'      search=[{searches}]  fusion={fusion}  rerank={rerank}  q_transform={qx}')
    print(f'      generation_model={model}')


Loaded 2 RAG pipeline configurations:
  - techqa_or_v1_baseline
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - techqa_or_v3_embed_bge
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct


## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [28]:
get_ipython().system('pip install rank_bm25 -q')

# Run experiments


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [29]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run['config'].name}: {run['total_written']} queries")

Running 2 configurations from 0 to 100 queries...
Parallel mode: False
Progress: 20/100 (20.0%) | QPS: 281496.91 | ETA: 0s | Elapsed: 0sUsing key #2: ****fbdbUsing key #2: ****fbdb

Progress: 20/100 (20.0%) | QPS: 9.89 | ETA: 8s | Elapsed: 2ssUsing key #2: ****fbdb
Progress: 21/100 (21.0%) | QPS: 5.21 | ETA: 15s | Elapsed: 4sUsing key #2: ****fbdb
Progress: 23/100 (23.0%) | QPS: 2.29 | ETA: 34s | Elapsed: 10sUsing key #2: ****fbdb
Progress: 23/100 (23.0%) | QPS: 0.95 | ETA: 1.3m | Elapsed: 24sUsing key #2: ****fbdb
Progress: 24/100 (24.0%) | QPS: 0.25 | ETA: 5.0m | Elapsed: 1.6mUsing key #2: ****fbdb
Progress: 25/100 (25.0%) | QPS: 0.23 | ETA: 5.3m | Elapsed: 1.8mUsing key #2: ****fbdb
Progress: 26/100 (26.0%) | QPS: 0.23 | ETA: 5.5m | Elapsed: 1.9mUsing key #2: ****fbdb
Progress: 27/100 (27.0%) | QPS: 0.20 | ETA: 6.0m | Elapsed: 2.2m

2026-07-25 16:07:28,136 ERROR rag.pipeline.rag_pipeline: Query failed: Expecting value: line 515 column 1 (char 2827)


Using key #2: ****fbdb
Progress: 28/100 (28.0%) | QPS: 0.20 | ETA: 5.9m | Elapsed: 2.3mUsing key #2: ****fbdb
Progress: 29/100 (29.0%) | QPS: 0.21 | ETA: 5.7m | Elapsed: 2.3mUsing key #2: ****fbdb
Progress: 30/100 (30.0%) | QPS: 0.20 | ETA: 5.7m | Elapsed: 2.4mUsing key #2: ****fbdb
Progress: 31/100 (31.0%) | QPS: 0.21 | ETA: 5.5m | Elapsed: 2.5mUsing key #2: ****fbdb
Progress: 32/100 (32.0%) | QPS: 0.21 | ETA: 5.4m | Elapsed: 2.5mUsing key #2: ****fbdb
Progress: 34/100 (34.0%) | QPS: 0.21 | ETA: 5.2m | Elapsed: 2.7mUsing key #2: ****fbdb
Progress: 35/100 (35.0%) | QPS: 0.21 | ETA: 5.1m | Elapsed: 2.7mUsing key #2: ****fbdb
Progress: 35/100 (35.0%) | QPS: 0.20 | ETA: 5.3m | Elapsed: 2.9mUsing key #2: ****fbdb
Progress: 36/100 (36.0%) | QPS: 0.21 | ETA: 5.2m | Elapsed: 2.9mUsing key #2: ****fbdb
Progress: 37/100 (37.0%) | QPS: 0.21 | ETA: 5.0m | Elapsed: 2.9mUsing key #2: ****fbdb
Progress: 38/100 (38.0%) | QPS: 0.21 | ETA: 4.9m | Elapsed: 3.0mUsing key #2: ****fbdb
Progress: 39/100 (39

## 7b. Evaluate Existing JSONL Files

Run offline evaluation on already-generated JSONL files.
Uses experiment-level evaluation config — all configs are scored with the same judge model.

- `parallel_runs=True` — evaluate multiple configs simultaneously
- `parallel_config_run=True` — evaluate records within each config in parallel

In [30]:
# Discover all configs and build run dicts from existing JSONL files
configs = runner.load_configs()
runs = []
for cfg in configs:
    jsonl_path = experiment_config.temp_dir / f"{cfg.name}.jsonl"
    if jsonl_path.exists():
        runs.append({"config_name": cfg.name, "config": cfg, "jsonl_path": jsonl_path})
    else:
        print(f"  Skipping {cfg.name} — no JSONL found")

print(f"Found {len(runs)} configs with JSONL files")

# Evaluate all configs: parallel across configs + parallel within each config
eval_runs = runner.evaluate_runs(
    runs,
    parallel_runs=True,
    parallel_config_run=True,
)

# Use eval_runs for report generation downstream
runs = eval_runs
print(f"\n✅ Evaluation complete: {len(eval_runs)} configs")

Found 2 configs with JSONL files
Using key #2: ****fbdb
Using key #2: ****fbdb
HTTP 402 (retryable) on ****fbdb
Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2000 tokens, but can only afford 1499. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2000 tokens, but can only afford 1499. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2000 tokens, but can only afford 1499. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 200

Record 28 evaluation failed: Judge did not return valid JSON:

```
{
  "relevance_explanation": "The question asks about the versions of WSAS supported with Oracle


Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462


Record 51 evaluation failed: Judge did not return valid JSON:

```
{
  "relevance_explanation": "The question asks about SSH connection failures after an upgrade to version 7.5.2 and above. The relevant documents discuss changes in default algorithms, disabled algorithms, and potential solutions to resolve the issue.",
  "all_relevant_sentence_keys": ["d2s5", "d2s6", "d2s7", "d2s8", "d2s9", "d3s0", "d3s1", "d3s2", "d3s3", "d3s4", "d3s5", "d3s6", "d3s7", "d3s8", "d3s9", "d3s10", "d1s4", "d1s5", "d1s6", "d1s7"],
  "overall_supported_explanation": "Most of the response sentences are supported by the documents, but some information is not explicitly mentioned or is partially supported.",
  "overall_supported": false,
  "sentence_support_information": [
    {
      "response_sentence_key": "a",
      "explanation": "The sentence is partially supported by d3s0, which mentions that OpenSSH 6.9p1 has changes in the default set of algorithms, but it does not specifically mention version 7.5.2."

Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462


...

Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462


Record 90 evaluation failed: Judge did not return valid JSON:

```
{
  "relevance_explanation": "The question asks for details regarding the Security Bulletin: Multiple vulnerabilities in IBM Java Runtime affect IBM Integration Bus and WebSphere Message Broker (CVE-2015-0138), also known as the FREAK attack. The relevant documents provide information on the vulnerability, affected products, and remediation fixes.",
  "all_relevant_sentence_keys": ["d0s0", "d0s1", "d0s2", "d0s3", "d0s


Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462


Record 88 evaluation failed: Expecting ',' delimiter: line 42 column 153 (char 999)


Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
Using key #3: ****2462
[techqa_or_v3_embed_bge] Evaluation complete → rag-experiments/techqa-openrouter-experiment/temp/techqa_or_v3_embed_bge.jsonl

✅ Evaluation complete: 2 configs


## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [31]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: rag-experiments/techqa-openrouter-experiment/reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [32]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    


Configuration: techqa_or_v1_baseline

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `techqa_or_v1_baseline`

**name**: techqa_or_v1_baseline  •  **mode**: test  •  **providers**: {'openrouter': {'type': 'openrouter', 'api_key_env': 'OPENROUTER_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'fixed_word', 'config': {'max_words': 200, 'overlap_words': 20}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 384}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 5}}]}, 'query_transform': None, 'fusion': None, 'rerank': None}  •  **generation**: {'strategy': 'default', 'provider': 'openrouter', 'config': {'model': 'meta-llama/llama-3.3-70b-instruct', 'temperature': 0.0, 'max_tokens': 750, 'system_prompt': 'You are an IBM technical-support question answering assistant.\nYour task is to answer questions using ONLY information from the retrieved\ntechnote passages.\n\nCRITICAL RULES:\n1. Answer ONLY from the passages provided. Do not use external knowledge.\n2. Include all relevant steps, error codes, product versions, APAR numbers,\n   and configuration details found in the passages.\n3. Preserve exact technical identifiers (version strings, file names,\n   parameter names, error messages) verbatim.\n4. Do NOT add phrases like "based on general knowledge" or hedge unnecessarily.\n5. If the passages do not contain enough information, respond with exactly:\n   "The passages do not provide sufficient information to answer this question."\n6. Every claim must be directly supported by the passages provided.\n', 'user_prompt': 'Passages:\n{context}\n\nQuestion: {query}\n\nAnswer (from passages only, include all relevant steps and technical details):\n'}}  •  **evaluation**: {'type': 'trace', 'provider': 'openrouter', 'config': {'model': 'meta-llama/llama-3.3-70b-instruct', 'temperature': 0.0, 'max_tokens': 2000}}  •  **cache**: {'enabled': True, 'cache_dir': './cache_openrouter_techqa'}  •  **start_index**: None  •  **end_index**: None  •  **logging_config**: {'enabled': True, 'level': 'INFO', 'show_progress': True}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,"Using cobol copybooks Sometimes, there will be errors/fields missing in type...","[{'text': '', 'title': '', 'content': 'TECHNOTE (TROUBLESHOOTING) PROBLEM(AB...","According to Document 2, when using COBOL copybooks, errors can occur if the...",0.3600,0.0117,0.3483,0.3000,0.0117,0.2883,0.6667,1.0000,-0.3333,0.0,0.0,0.0
1,Is WebSphere Transformation Extender (WTX) supported for IBM Integration Bus...,"[{'text': '', 'title': '', 'content': 'WebSphere Transformation Extender; We...",The passages do not provide sufficient information to answer this question.,0.0000,0.0270,-0.0270,0.0000,0.0270,-0.0270,0.0000,1.0000,-1.0000,1.0,1.0,0.0
2,Want to find out if Microsoft Edge is supported with ICC? Want to find out M...,"[{'text': '', 'title': '', 'content': 'THE PROBLEM Update the video driver t...","According to the passages, Microsoft Edge is supported with IBM Content Coll...",0.1250,0.0027,0.1223,0.0833,0.0013,0.0820,0.6667,0.5000,0.1667,1.0,1.0,0.0
3,What IBM Business Process Manager version is affected by the Apache Commons ...,"[{'text': '', 'title': '', 'content': 'SECURITY BULLETIN SUMMARY Vulnerabili...",The passages do not provide sufficient information to answer this question. ...,0.3462,0.0000,0.3462,0.1923,0.0000,0.1923,0.5556,1.0000,-0.4444,1.0,1.0,0.0
4,Error while installing SpSS modeler trial version I have downloaded the SPSS...,"[{'text': '', 'title': '', 'content': 'TECHNOTE (TROUBLESHOOTING) PROBLEM(AB...",The error 1152 during extraction of the Modeler self-extracting executable i...,0.1455,0.0833,0.0622,0.1091,0.0463,0.0628,0.7500,0.5556,0.1944,0.0,1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Why are the users unable to login to Web GUI 8.1 after I modify timeout to 2...,"[{'text': '', 'title': '', 'content': 'DASHL2; session; time-out; timeout; c...",The users are unable to login to DASH after modifying the timeout to 2147483...,0.3043,0.0244,0.2799,0.1522,0.0244,0.1278,0.5000,1.0000,-0.5000,0.0,0.0,0.0
96,When and why Apache Storm process of ITNM creating db2 files in /tmp? \n\nWh...,"[{'text': '', 'title': '', 'content': 'db2 ; libdb2.so.1 TECHNOTE (TROUBLESH...",The passages do not provide sufficient information to answer this question.,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,-1.0000,1.0,0.0,1.0
97,Why are some Datatypes corrupt after Impact cluster failover/failback ? \n\n...,"[{'text': '', 'title': '', 'content': 'netcool; impact; netcool/impact; nci;...",The passages do not provide sufficient information to directly answer the qu...,0.3750,0.0591,0.3159,0.1250,0.0000,0.1250,0.3333,0.0000,0.3333,0.0,0.0,0.0
98,Cannot launch RFT 8.2 \n\nHello.\n\nCould anyone can help to see why I canno...,"[{'text': '', 'title': '', 'content': 'TECHNOTE (TROUBLESHOOTING) PROBLEM(AB...",The passages do not provide sufficient information to answer this question.,0.3659,0.0246,0.3413,0.0000,0.0296,-0.0296,0.0000,1.0000,-1.0000,1.0,1.0,0.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.3161,0.0703,0.2770,0.1039,0.2590
1,utilization_score,0.1474,0.0259,0.2043,0.0401,0.1335
2,completeness_score,0.3309,0.5816,0.3539,0.3887,0.4690
3,adherence_score,0.3918,0.5300,0.4881,0.4991,0.6392


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [33]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: rag-experiments/techqa-openrouter-experiment/reports/comparison.csv


In [34]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,techqa_or_v5_hybrid_rrf,0.1321,0.0987,0.0724,0.0603,0.4514,0.5174,0.2500,0.4000
1,techqa_or_v4_chunk_sentence,0.4377,0.4123,0.1659,0.1647,0.2614,0.6025,0.5500,0.4000
2,techqa_or_v10_hyde_match,0.2730,0.2292,0.0833,0.0652,0.2808,0.4716,0.4706,0.5294
3,techqa_or_v7_stepback,0.3685,0.3265,0.1156,0.0965,0.3103,0.5345,0.3158,0.5263
4,techqa_or_v6_rerank_only,0.4120,0.3781,0.1931,0.1778,0.3655,0.5381,0.2000,0.4500
5,techqa_or_v2_hybrid_wsum,0.1361,0.1093,0.0714,0.0687,0.3607,0.5377,0.3000,0.4500
6,techqa_or_v3_embed_bge,0.3161,0.2590,0.1474,0.1335,0.3309,0.4690,0.3918,0.6392
7,techqa_or_v1_baseline,0.3529,0.2983,0.1572,0.1467,0.3001,0.5108,0.4681,0.5851
8,techqa_or_v9_bge_sentence,0.5578,0.5266,0.2155,0.2078,0.3161,0.4998,0.4000,0.5500
9,techqa_or_v8_best,0.4402,0.3993,0.2203,0.1998,0.4565,0.4468,0.2500,0.6000


## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons